<a href="https://colab.research.google.com/github/puongmint/TracyNguyen_Portfolio/blob/main/amazon_product_recommender_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Amazon Product Recommender System

This project builds a scalable product recommendation system using over **436,000 Amazon products**.

The system allows users to search for a product and receive recommendations based on product similarity and popularity signals.

### Objectives
- Clean and prepare product data from multiple source files
- Build a content-based recommender using TF-IDF and cosine similarity
- Improve ranking with a simple hybrid recommendation approach
- Visualise recommendations using product cards with real images

### Key Features
- Large-scale dataset (436k+ products)
- Content-based recommendation using TF-IDF
- Fast similarity retrieval with Nearest Neighbors
- Hybrid ranking using ratings and review counts
- Interactive demo with product images
## System Pipeline
Dataset → Cleaning → Feature Engineering → TF-IDF Vectorization  
→ Nearest Neighbor Search → Hybrid Ranking → Interactive Recommendation Demo
### Technologies
Python • Pandas • Scikit-learn • NLP • Recommender Systems • Jupyter Widgets

In [ ]:
import os
import glob
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, HTML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

Download dataset directly from Kaggle (Google Colab)

In [ ]:
# Download Amazon Products Dataset from Kaggle

!pip install -q kaggle

from google.colab import files
import os

print("Step 1: Please upload your kaggle.json API key")
files.upload()

print("Step 2: Configuring Kaggle API...")

os.makedirs("/root/.kaggle", exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

print("Step 3: Downloading dataset from Kaggle...")

!kaggle datasets download -d lokeshparab/amazon-products-dataset

print("Step 4: Extracting dataset...")

!unzip -o amazon-products-dataset.zip > /dev/null

print("✅ Dataset downloaded and ready to use.")

Load all CSV files

In [ ]:
data_path = "/content"
csv_files = glob.glob(os.path.join(data_path, "*.csv"))

df_list = []

for file in csv_files:
    try:
        temp = pd.read_csv(file)
        temp["source_file"] = os.path.basename(file)
        df_list.append(temp)
    except Exception as e:
        print(f"Could not load {file}: {e}")

if not df_list:
    raise ValueError("No CSV files were loaded. Please check your dataset path.")

df = pd.concat(df_list, ignore_index=True)

print("Number of CSV files loaded:", len(df_list))
print("Combined dataset shape:", df.shape)
display(df.head())

In [ ]:
# Standardise schema for this dataset
title_col = "name"
category_col = "main_category"
brand_col = None
price_col = "discount_price"
rating_col = "ratings"
review_count_col = "no_of_ratings"
description_col = "sub_category"
image_col = "image"
url_col = "link"

work_df = df.copy()

selected_cols = [
    title_col,
    category_col,
    price_col,
    rating_col,
    review_count_col,
    description_col,
    image_col,
    url_col,
    "source_file"
]

if brand_col is not None:
    selected_cols.append(brand_col)

selected_cols = [col for col in selected_cols if col in work_df.columns]
work_df = work_df[selected_cols].copy()

rename_map = {
    title_col: "title",
    category_col: "category",
    price_col: "price",
    rating_col: "rating",
    review_count_col: "review_count",
    description_col: "description",
    image_col: "image",
    url_col: "url"
}

if brand_col is not None and brand_col in work_df.columns:
    rename_map[brand_col] = "brand"

work_df.rename(columns=rename_map, inplace=True)

if "brand" not in work_df.columns:
    work_df["brand"] = ""

print("Columns after standardisation:")
print(work_df.columns.tolist())
display(work_df.head())

In [ ]:
# Helper functions

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def clean_price(value):
    if pd.isna(value):
        return np.nan
    value = str(value).replace("₹", "").replace("$", "").replace("£", "").replace(",", "").strip()
    match = re.search(r"\d+(\.\d+)?", value)
    return float(match.group()) if match else np.nan

def clean_rating(value):
    if pd.isna(value):
        return np.nan

    value = str(value)

    # extract decimal number
    match = re.search(r"\d+(\.\d+)?", value)

    if match:
        rating = float(match.group())

        # keep only valid Amazon ratings
        if 0 <= rating <= 5:
            return rating

    return np.nan


work_df["rating_num"] = work_df["rating"].apply(clean_rating)

# replace missing ratings
work_df["rating_num"] = work_df["rating_num"].fillna(0)

def clean_numeric(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    # remove commas
    x = x.replace(",", "")

    # extract numbers
    match = re.search(r"\d+", x)

    if match:
        return int(match.group())

    return np.nan

Data Preprocessing

In [ ]:
for col in ["title", "category", "brand", "description"]:
    work_df[col] = work_df[col].fillna("").astype(str)

work_df["price_num"] = work_df["price"].apply(clean_price)
work_df["rating_num"] = work_df["rating"].apply(clean_rating)
work_df["review_count_num"] = work_df["review_count"].apply(clean_numeric)

work_df["title_clean"] = work_df["title"].apply(clean_text)
work_df["category_clean"] = work_df["category"].apply(clean_text)
work_df["brand_clean"] = work_df["brand"].apply(clean_text)
work_df["description_clean"] = work_df["description"].apply(clean_text)

# keep one version of duplicated title + price combinations
work_df.drop_duplicates(subset=["title", "price"], inplace=True)
work_df.reset_index(drop=True, inplace=True)

# fill missing numeric values
work_df["price_num"] = work_df["price_num"].fillna(work_df["price_num"].median())
work_df["rating_num"] = work_df["rating_num"].fillna(work_df["rating_num"].median())
work_df["review_count_num"] = work_df["review_count_num"].fillna(0)

# optional indicators for missingness
work_df["price_missing"] = work_df["price"].isna().astype(int)
work_df["rating_missing"] = work_df["rating"].isna().astype(int)
work_df["review_missing"] = work_df["review_count"].isna().astype(int)

print("Cleaned dataset shape:", work_df.shape)
display(work_df.head())

In [ ]:
# Missing value summary


missing_summary = pd.DataFrame({
    "missing_count": work_df.isna().sum(),
    "missing_percent": (work_df.isna().sum() / len(work_df) * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary)

In [ ]:
# Handle missing numeric values

# convert numeric fields
work_df["price_num"] = work_df["price"].apply(clean_price)
work_df["rating_num"] = work_df["rating"].apply(clean_numeric)
work_df["review_count_num"] = work_df["review_count"].apply(clean_numeric)
work_df["desc_length"] = work_df["description"].fillna("").apply(lambda x: len(x.split()))

# fill missing values
work_df["price_num"] = work_df["price_num"].fillna(work_df["price_num"].median())

# products with no rating → treat as neutral
work_df["rating_num"] = work_df["rating_num"].fillna(0)

# products with no reviews → treat as zero popularity
work_df["review_count_num"] = work_df["review_count_num"].fillna(0)

print("Numeric fields cleaned.")

In [ ]:
# Clean image column

work_df["image"] = work_df["image"].fillna("").astype(str)

# keep first image if multiple links exist
work_df["image"] = work_df["image"].str.split("|").str[0].str.strip()

# blank / invalid values -> NaN
invalid_image_values = ["", "nan", "none", "null", "na"]
work_df.loc[work_df["image"].str.lower().isin(invalid_image_values), "image"] = np.nan

# keep only likely URLs
work_df.loc[~work_df["image"].fillna("").str.startswith("http"), "image"] = np.nan

print("Valid images:", work_df["image"].notna().sum())

Exploratory Data Analysis

In [ ]:
# EDA 1: Dataset overview

print("Dataset shape:", work_df.shape)

summary_missing = pd.DataFrame({
    "missing_count": work_df[["price", "rating", "review_count"]].isna().sum(),
    "missing_percent": (work_df[["price", "rating", "review_count"]].isna().sum() / len(work_df) * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(summary_missing)

print("\nClean numeric columns check:")
print("price_num missing:", work_df["price_num"].isna().sum())
print("rating_num missing:", work_df["rating_num"].isna().sum())
print("review_count_num missing:", work_df["review_count_num"].isna().sum())

### Dataset Overview
The dataset contains over 436,000 products, providing a strong basis for building a large-scale product recommendation system.

Missing values are concentrated in rating, review count, and price fields, which is common in marketplace data where many products are newly listed or have limited customer feedback. Clean numeric versions of these variables were created to support downstream modelling and analysis.

In [ ]:
# EDA 2: Top categories

plt.figure(figsize=(10, 6))

top_categories = work_df["category"].value_counts().head(15)
top_categories.sort_values().plot(kind="barh")

plt.title("Top 15 Product Categories")
plt.xlabel("Number of Products")
plt.ylabel("Category")
plt.grid(axis="x", linestyle="--", alpha=0.4)

plt.show()

### Category Distribution Insight
The dataset is concentrated in categories such as accessories, men's clothing, and women's clothing. This suggests that the product catalog is heavily weighted toward fashion and lifestyle items.

From a recommendation perspective, categories with more products are likely to produce stronger recommendations because the model has more similar items to compare.

In [ ]:
# EDA 3: Rating distribution

plt.figure(figsize=(8, 5))

plt.hist(work_df["rating_num"], bins=20)

plt.title("Distribution of Product Ratings")
plt.xlabel("Rating")
plt.ylabel("Frequency")
plt.xlim(0, 5)

plt.show()

### Product Rating Distribution Insight
Most rated products fall between 3.5 and 4.5, which is typical for e-commerce platforms where customer feedback is generally skewed toward positive ratings.

A large number of products also have a rating of 0, representing products with no ratings yet. This reinforces the importance of relying on product content features, not just feedback signals, in the recommendation system.

In [ ]:
# EDA 4: Review count distribution

plt.figure(figsize=(8, 5))

plt.hist(np.log1p(work_df["review_count_num"]), bins=30)

plt.title("Distribution of Review Counts (log scale)")
plt.xlabel("log(1 + review_count)")
plt.ylabel("Frequency")

plt.show()

### Review Count Distribution Insight
Review counts follow a long-tail distribution: most products have very few reviews, while a small number of products receive very large amounts of customer feedback.

This pattern is common in e-commerce platforms and suggests that popularity signals are unevenly distributed across the catalog.

In [ ]:
# EDA 5: Price distribution

plt.figure(figsize=(8, 5))

plt.hist(np.log1p(work_df["price_num"]), bins=40)

plt.title("Log Distribution of Product Prices")
plt.xlabel("log(1 + price)")
plt.ylabel("Frequency")

plt.show()

### Price Distribution Insight
Product prices are right-skewed, with most items concentrated in lower to mid-price ranges and a smaller set of premium products extending the upper tail.

This reflects the typical structure of online retail platforms, where affordable products dominate the catalog but high-value items still contribute meaningful diversity.

In [ ]:
# EDA 6: Description length vs rating

sample_df = work_df.sample(10000, random_state=42)

plt.figure(figsize=(8, 6))

plt.scatter(
    sample_df["desc_length"],
    sample_df["rating_num"],
    alpha=0.25
)

plt.title("Description Length vs Rating")
plt.xlabel("Description Length")
plt.ylabel("Rating")

plt.show()

### Description Length vs Rating

The scatter plot shows no strong relationship between product description length and customer rating.

While longer descriptions may provide more detailed information to customers, they do not necessarily result in higher ratings. However, richer textual descriptions can still improve the performance of the content-based recommendation system by providing more informative features for similarity calculations.

In [ ]:
# EDA 7: Rating vs review count

sample_df = work_df.sample(10000, random_state=42)

plt.figure(figsize=(8,6))

plt.scatter(
    sample_df["rating_num"],
    np.log1p(sample_df["review_count_num"]),
    alpha=0.3
)

plt.title("Rating vs Review Count (log scale)")
plt.xlabel("Rating")
plt.ylabel("log(1 + Review Count)")

plt.show()

### Rating vs Review Count

The relationship between product ratings and review counts reveals a typical long-tail engagement pattern. Most products receive relatively few reviews, while a small number of highly popular products accumulate very large numbers of reviews.

Highly reviewed products tend to cluster around ratings between 4 and 4.5, suggesting that popular products are often well received by customers.

However, a large number of products have few or no reviews, highlighting the sparsity of user feedback signals in the dataset. This supports the use of content-based recommendation methods that rely on product descriptions, titles, and categories rather than only user interaction data.

## Key Insights

Several insights from the data informed the recommendation design:

• Ratings are heavily skewed toward higher scores (4-5 stars).  
• Review counts vary widely, indicating popularity differences between products.  
• Product titles and descriptions contain rich textual signals useful for similarity-based recommendations.

These observations support using **text-based similarity combined with popularity signals**.

## Recommendation Approach

A **content-based recommendation system** was implemented using textual product information.

Steps:

1. Combine product title, category, brand, and description into a single text field.
2. Convert text into numerical features using **TF-IDF vectorization**.
3. Use **Nearest Neighbors (cosine similarity)** to retrieve similar products.
4. Apply a **hybrid ranking score** combining:

- similarity score
- product rating
- review count

This improves recommendation quality by balancing relevance and popularity.

Feature engineering: combined text

In [ ]:
work_df["combined_text"] = (
    work_df["title_clean"] + " " +
    work_df["title_clean"] + " " +   # repeat title for stronger weight
    work_df["category_clean"] + " " +
    work_df["description_clean"]
)

print("Example combined text:")
print(work_df["combined_text"].iloc[0][:500])

In [ ]:
# TF-IDF vectorization

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    dtype=np.float32
)

tfidf_matrix = tfidf.fit_transform(work_df["combined_text"])

print("TF-IDF matrix shape:", tfidf_matrix.shape)

### TF-IDF Feature Representation

Product text fields (title, category, and description) were combined into a single text feature and transformed using TF-IDF vectorization.

The resulting feature matrix has a shape of:

436,119 products × 20,000 textual features.

Each product is represented as a high-dimensional vector capturing the importance of words appearing in its description and title.

Cosine similarity is then used to measure similarity between product vectors and generate recommendations based on textual content.

In [ ]:
# Fast nearest-neighbors index

nn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=150
)

nn_model.fit(tfidf_matrix)

print("NearestNeighbors index built.")

### Recommendation Search Optimisation

To improve scalability, a `NearestNeighbors` index was built on top of the TF-IDF matrix using cosine distance.

This avoids repeatedly computing similarity scores against the entire product catalog for every query and makes recommendation retrieval much faster on a dataset of over 436,000 products.

In [ ]:
# Build title lookup index

work_df["title_key"] = work_df["title"].astype(str).str.strip().str.lower()

indices = pd.Series(work_df.index, index=work_df["title_key"])
indices = indices[~indices.index.duplicated(keep="first")]

print("Indexed products:", len(indices))

In [ ]:
total_rows = len(work_df)
unique_titles = work_df["title_key"].nunique()

print("Total rows:", total_rows)
print("Unique title keys:", unique_titles)
print("Duplicate title keys removed:", total_rows - unique_titles)

### Product Lookup Index

A title-based lookup index was created to support fast recommendation queries.

After normalizing product titles and removing duplicate title keys, the final index contains 395,146 unique products. The difference between total rows and indexed products reflects duplicate or repeated product titles in the raw dataset.

This step ensures stable title-based lookup while avoiding ambiguity during similarity search.

In [ ]:
# Search function

def search_products(keyword, n=10, require_image=False):
    keyword_clean = clean_text(keyword)

    mask = (
        work_df["title_clean"].str.contains(keyword_clean, na=False, regex=False) |
        work_df["category_clean"].str.contains(keyword_clean, na=False, regex=False) |
        work_df["description_clean"].str.contains(keyword_clean, na=False, regex=False)
    )

    results = work_df[mask].copy()

    if require_image:
        results = results[results["image"].notna()].copy()

    if results.empty:
        print(f"No products found for keyword: {keyword}")
        return pd.DataFrame()

    results["search_score"] = (
        results["title_clean"].str.contains(keyword_clean, na=False, regex=False).astype(int) * 4 +
        results["category_clean"].str.contains(keyword_clean, na=False, regex=False).astype(int) * 2 +
        results["description_clean"].str.contains(keyword_clean, na=False, regex=False).astype(int) * 1
    )

    cols_to_show = ["title", "category", "price", "rating", "review_count", "image", "url"]
    cols_to_show = [col for col in cols_to_show if col in results.columns]

    return (
        results.sort_values(
            ["search_score", "review_count_num", "rating_num"],
            ascending=False
        )[cols_to_show]
        .head(n)
        .reset_index(drop=True)
    )

In [ ]:
# Fast content-based recommendation function

def recommend_products(product_title, top_n=10):
    product_title_lower = str(product_title).strip().lower()

    if product_title_lower not in indices.index:
        return pd.DataFrame()

    idx = int(indices[product_title_lower])

    distances, neighbor_idx = nn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=top_n + 20
    )

    distances = distances.flatten()
    neighbor_idx = neighbor_idx.flatten()

    candidates = work_df.iloc[neighbor_idx].copy()
    candidates["similarity_score"] = 1 - distances

    # remove the exact same product
    candidates = candidates[
        candidates["title"].str.lower().str.strip() != product_title_lower
    ].copy()

    # remove near-duplicate title variants
    candidates["title_dedup"] = (
        candidates["title"]
        .astype(str)
        .str.lower()
        .str.replace(r"\(.*?\)", "", regex=True)
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    candidates = candidates.drop_duplicates(subset="title_dedup")

    cols_to_show = ["title", "category", "brand", "price", "rating", "review_count", "similarity_score"]
    cols_to_show = [col for col in cols_to_show if col in candidates.columns]

    return (
        candidates.sort_values("similarity_score", ascending=False)[cols_to_show]
        .head(top_n)
        .reset_index(drop=True)
    )

In [ ]:
# Fast hybrid recommendation function

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

work_df["rating_scaled"] = scaler.fit_transform(work_df[["rating_num"]])
work_df["review_scaled"] = scaler.fit_transform(work_df[["review_count_num"]])

def recommend_products_hybrid(product_title, top_n=10, alpha=0.8, beta=0.1, gamma=0.1):
    product_title_lower = str(product_title).strip().lower()

    if product_title_lower not in indices.index:
        return pd.DataFrame()

    idx = int(indices[product_title_lower])

    distances, neighbor_idx = nn_model.kneighbors(
        tfidf_matrix[idx],
        n_neighbors=150
    )

    distances = distances.flatten()
    neighbor_idx = neighbor_idx.flatten()

    candidates = work_df.iloc[neighbor_idx].copy()
    candidates["similarity_score"] = 1 - distances

    # remove exact same product
    candidates = candidates[
        candidates["title"].str.lower().str.strip() != product_title_lower
    ].copy()

    candidates["hybrid_score"] = (
        alpha * candidates["similarity_score"] +
        beta * candidates["rating_scaled"] +
        gamma * candidates["review_scaled"]
    )

    # remove near-duplicate title variants
    candidates["title_dedup"] = (
        candidates["title"]
        .astype(str)
        .str.lower()
        .str.replace(r"\(.*?\)", "", regex=True)
        .str.replace(r"[^a-z0-9\s]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    candidates = candidates.drop_duplicates(subset="title_dedup")

    cols_to_show = [
        "title", "category", "brand", "price", "rating",
        "review_count", "similarity_score", "hybrid_score"
    ]
    cols_to_show = [col for col in cols_to_show if col in candidates.columns]

    return (
        candidates.sort_values("hybrid_score", ascending=False)[cols_to_show]
        .head(top_n)
        .reset_index(drop=True)
    )

## Evaluation

To evaluate recommendation quality, a **Category Match Rate@5** metric was used.

This measures how often the recommended products belong to the same category as the query product.

Result:

Category Match Rate@5 ≈ 0.94

This indicates that the system retrieves highly relevant products based on product similarity.

In [ ]:
# Evaluation

def category_match_rate(sample_size=100, k=5, random_state=42):
    if "category" not in work_df.columns:
        print("No 'category' column found in work_df.")
        return None

    rng = np.random.RandomState(random_state)
    sample_indices = rng.choice(len(work_df), min(sample_size, len(work_df)), replace=False)

    match_scores = []

    for idx in sample_indices:
        title = work_df.iloc[idx]["title"]
        true_category = work_df.iloc[idx]["category"]

        recs = recommend_products(title, top_n=k)

        if isinstance(recs, pd.DataFrame) and "category" in recs.columns and len(recs) > 0:
            matches = (recs["category"] == true_category).sum()
            match_scores.append(matches / k)

    if not match_scores:
        return None

    return np.mean(match_scores)

score = category_match_rate(sample_size=100, k=5)

if score is not None:
    print(f"Category match rate@5: {score:.2f}")
else:
    print("Category match rate could not be calculated.")

### Recommendation Evaluation

The recommendation system was evaluated using a category consistency metric.  
For each product, the model generates the top 5 most similar items, and the proportion of recommendations belonging to the same category is calculated.

The model achieved a **Category Match Rate@5 of 0.94**, meaning that on average 94% of recommended products belong to the same category as the query item.

This indicates that the TF-IDF based similarity approach effectively captures semantic relationships between products and produces category-consistent recommendations.

While category match rate provides a useful sanity check for recommendation quality, it does not measure real user relevance. In production systems, additional metrics such as click-through rate, conversion rate, or user interaction data would be required.

In [ ]:
# Product card display using cleaned columns

from IPython.display import display, HTML
import pandas as pd

def get_image_url(url):
    if pd.isna(url):
        return "https://via.placeholder.com/200x200?text=No+Image"

    url = str(url).strip()
    if url == "" or not url.startswith("http"):
        return "https://via.placeholder.com/200x200?text=No+Image"

    return url


def show_product_cards(data, title="Products", n=5):
    if data is None or len(data) == 0:
        display(HTML(f"<h3>{title}</h3><p>No products to display.</p>"))
        return

    html = f"""
    <h2 style="margin-bottom:16px;">{title}</h2>
    <div style="display:flex; flex-wrap:wrap; gap:16px;">
    """

    for _, row in data.head(n).iterrows():
        product_title = str(row["title"]) if "title" in row and pd.notna(row["title"]) else "Unknown Product"
        category = str(row["category"]) if "category" in row and pd.notna(row["category"]) else "N/A"
        price = str(row["price"]) if "price" in row and pd.notna(row["price"]) else "N/A"
        rating = str(row["rating"]) if "rating" in row and pd.notna(row["rating"]) else "N/A"
        reviews = str(row["review_count"]) if "review_count" in row and pd.notna(row["review_count"]) else "N/A"
        image_url = get_image_url(row["image"] if "image" in row else None)
        product_url = str(row["url"]) if "url" in row and pd.notna(row["url"]) and str(row["url"]).strip() != "" else "#"
        similarity = row["similarity_score"] if "similarity_score" in row and pd.notna(row["similarity_score"]) else None
        hybrid = row["hybrid_score"] if "hybrid_score" in row and pd.notna(row["hybrid_score"]) else None

        extra_html = ""
        if similarity is not None:
            extra_html += f"""
            <div style="font-size:12px; color:#444; margin-bottom:4px;">
                <b>Similarity:</b> {float(similarity):.3f}
            </div>
            """
        if hybrid is not None:
            extra_html += f"""
            <div style="font-size:12px; color:#444; margin-bottom:4px;">
                <b>Hybrid score:</b> {float(hybrid):.3f}
            </div>
            """

        html += f"""
        <div style="
            width:250px;
            border:1px solid #e0e0e0;
            border-radius:14px;
            padding:14px;
            background:#ffffff;
            box-shadow:0 2px 8px rgba(0,0,0,0.08);
        ">
            <div style="
                height:180px;
                display:flex;
                align-items:center;
                justify-content:center;
                margin-bottom:12px;
                background:#f7f7f7;
                border-radius:10px;
                overflow:hidden;
            ">
                <img src="{image_url}" alt="{product_title}"
                     style="max-width:100%; max-height:100%; object-fit:contain; border-radius:10px;"
                     onerror="this.src='https://via.placeholder.com/200x200?text=No+Image';" />
            </div>

            <div style="font-size:14px; font-weight:600; margin-bottom:8px; line-height:1.4; min-height:58px;">
                {product_title[:110]}
            </div>

            <div style="font-size:12px; color:#666; margin-bottom:6px;">
                <b>Category:</b> {category}
            </div>

            <div style="font-size:13px; margin-bottom:4px;">
                <b>Price:</b> {price}
            </div>

            <div style="font-size:13px; margin-bottom:4px;">
                <b>Rating:</b> {rating}
            </div>

            <div style="font-size:12px; color:#666; margin-bottom:8px;">
                <b>Reviews:</b> {reviews}
            </div>

            {extra_html}

            <a href="{product_url}" target="_blank"
               style="display:inline-block; padding:8px 12px; background:#111; color:white;
                      text-decoration:none; border-radius:8px; font-size:12px;">
               View Product
            </a>
        </div>
        """

    html += "</div>"
    display(HTML(html))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# controls
search_box = widgets.Text(
    value="shoes",
    description="Search:",
    placeholder="Type product keyword...",
    layout=widgets.Layout(width="50%")
)

search_button = widgets.Button(
    description="Search",
    button_style="primary"
)

# separate outputs
search_output = widgets.Output()
recommendation_output = widgets.Output()

def run_recommendation(change):
    with recommendation_output:
        clear_output(wait=True)

        selected_product = product_selector.value

        selected_df = work_df[work_df["title"] == selected_product][
            ["title", "category", "price", "rating", "review_count", "image", "url"]
        ].head(1)

        show_product_cards(selected_df, title="Selected Product", n=1)

        recs = recommend_products_hybrid(selected_product, top_n=5)

        if len(recs) > 0:
            recs_cards = pd.merge(
                recs,
                work_df[["title", "image", "url"]],
                on="title",
                how="left"
            )

            show_product_cards(recs_cards, title="Recommended Products", n=5)


def run_search(b):
    global product_selector

    with search_output:
        clear_output(wait=True)

        keyword = search_box.value.strip()
        print(f"Search results for: {keyword}")

        search_results = search_products(keyword, n=10)

        if len(search_results) == 0:
            print("No products found.")
            return

        show_product_cards(search_results, title=f"Search Results for '{keyword}'", n=5)

        product_selector = widgets.Dropdown(
            options=search_results["title"].tolist(),
            description="Select product:",
            layout=widgets.Layout(width="80%")
        )

        display(product_selector)

        # clear old observers if rerun
        product_selector.observe(run_recommendation, names="value")

    # clear old recommendations when new search runs
    with recommendation_output:
        clear_output(wait=True)


search_button.on_click(run_search)

display(search_box, search_button)
display(search_output)
display(recommendation_output)

# Conclusion

This project demonstrates how a content-based recommendation system can be built using product metadata and natural language processing techniques.

## Key outcomes
- Built a recommendation engine using TF-IDF and cosine similarity
- Improved ranking with a hybrid score using similarity, rating, and review count
- Created a visual recommendation interface with real product images
- Evaluated recommendation quality using category match rate

## Limitations
- Recommendations depend on product metadata quality
- No user behaviour or collaborative filtering was included
- Evaluation is simple and can be improved with stronger metrics

## Future Improvements

- Incorporate user behavior data (clicks, purchases)  
- Implement collaborative filtering  
- Add recommendation diversity optimization  
- Deploy the system using Streamlit for a full web interface  